Charger les données depuis le Volume

In [0]:
# Charger les données sauvegardées dans Volume
reviews = spark.read.parquet(
    "/Volumes/workspace/default/ecommerce_data/reviews_beauty"
)
metadata = spark.read.parquet(
    "/Volumes/workspace/default/ecommerce_data/metadata_beauty"
)

print(f" Reviews chargées   : {reviews.count():,} lignes")
print(f" Metadata chargées  : {metadata.count():,} lignes")

# Vérifier les colonnes disponibles
print(f"\nColonnes reviews  : {reviews.columns}")
print(f"Colonnes metadata : {metadata.columns}")

 Reviews chargées   : 701,528 lignes
 Metadata chargées  : 112,590 lignes

Colonnes reviews  : ['rating', 'title', 'text', 'images', 'asin', 'parent_asin', 'user_id', 'timestamp', 'helpful_vote', 'verified_purchase']
Colonnes metadata : ['main_category', 'title', 'average_rating', 'rating_number', 'features', 'description', 'price', 'images', 'videos', 'store', 'categories', 'details', 'parent_asin', 'bought_together', 'subtitle', 'author']


Nettoyer les données

In [0]:
from pyspark.sql.functions import col, when

# Supprimer les lignes avec des valeurs manquantes essentielles
reviews_clean = reviews.dropna(subset=["user_id", "parent_asin", "rating"])

# Garder seulement les ratings valides (entre 1 et 5)
reviews_clean = reviews_clean.filter(
    (col("rating") >= 1) & (col("rating") <= 5)
)

print(f"Avant nettoyage : {reviews.count():,}")
print(f"Après nettoyage : {reviews_clean.count():,}")
print(f"Lignes supprimées : {reviews.count() - reviews_clean.count():,}")

Avant nettoyage : 701,528
Après nettoyage : 701,528
Lignes supprimées : 0


 Convertir les IDs en entiers 

In [0]:
from pyspark.ml.feature import StringIndexer
from pyspark.sql.functions import col

# Créer les indexeurs
user_indexer = StringIndexer(inputCol="user_id", outputCol="user_idx")
item_indexer = StringIndexer(inputCol="parent_asin", outputCol="item_idx")

# Entraîner les indexeurs sur les données
user_model = user_indexer.fit(reviews_clean)
item_model = item_indexer.fit(reviews_clean)

# Appliquer la transformation
ratings_indexed = user_model.transform(reviews_clean)
ratings_indexed = item_model.transform(ratings_indexed)

# Convertir en entiers (ALS l'exige absolument)
ratings_indexed = ratings_indexed \
    .withColumn("user_idx", col("user_idx").cast("int")) \
    .withColumn("item_idx", col("item_idx").cast("int"))

print(" Conversion effectuée !")
print(f"Exemple : user_id 'ABC123' → user_idx {ratings_indexed.select('user_idx').first()[0]}")
display(ratings_indexed.select("user_id", "user_idx", "parent_asin", "item_idx", "rating").limit(5))

 Conversion effectuée !
Exemple : user_id 'ABC123' → user_idx 33934


user_id,user_idx,parent_asin,item_idx,rating
AGKHLEW2SOWHNMFQIJGBECAF7INQ,33934,B00YQ6X8EO,722,5
AGKHLEW2SOWHNMFQIJGBECAF7INQ,33934,B081TJ8YS3,9669,4
AE74DYR3QUGVPZJ3P7RFWBGIX7XQ,71462,B097R46CSY,4203,5
AFQLNQNQYFWQZPJQZS6V3NZU4QBQ,26005,B09JS339BZ,109899,1
AFQLNQNQYFWQZPJQZS6V3NZU4QBQ,26005,B08BZ63GMJ,42259,5


Sauvegarder les mappings

In [0]:
import json

# Ces mappings sont CRUCIAUX
# Sans eux, impossible de retrouver les vrais IDs après modélisation
user_mapping = user_model.labels  # index 0 → user_id original
item_mapping = item_model.labels  # index 0 → product_id original

print(f" Mapping utilisateurs : {len(user_mapping):,} users")
print(f" Mapping produits     : {len(item_mapping):,} produits")
print(f"\nExemple user_mapping[0] = '{user_mapping[0]}'")
print(f"Exemple item_mapping[0] = '{item_mapping[0]}'")

# Sauvegarder les mappings en JSON dans le Volume
user_mapping_path = "/Volumes/workspace/default/ecommerce_data/user_mapping.json"
item_mapping_path = "/Volumes/workspace/default/ecommerce_data/item_mapping.json"

with open(user_mapping_path, "w") as f:
    json.dump(list(user_mapping), f)

with open(item_mapping_path, "w") as f:
    json.dump(list(item_mapping), f)

print(f"\n Mappings sauvegardés dans le Volume !")

 Mapping utilisateurs : 631,986 users
 Mapping produits     : 112,565 produits

Exemple user_mapping[0] = 'AG73BVBKUOH22USSFJA5ZWL7AKXA'
Exemple item_mapping[0] = 'B085BB7B1M'

 Mappings sauvegardés dans le Volume !


Construire les profils utilisateurs

In [0]:
from pyspark.sql.functions import count, avg, stddev, max as spark_max, min as spark_min

# Profil de chaque utilisateur
user_profiles = ratings_indexed.groupBy("user_id", "user_idx").agg(
    count("*").alias("total_ratings"),           # combien de produits notés
    avg("rating").alias("avg_rating"),            # note moyenne donnée
    stddev("rating").alias("std_rating"),         # variabilité des notes
    spark_max("rating").alias("max_rating"),      # note max donnée
    spark_min("rating").alias("min_rating"),      # note min donnée
    spark_max("timestamp").alias("last_activity") # dernière activité
)

print(f" Profils construits pour {user_profiles.count():,} utilisateurs")
display(user_profiles.limit(10))

 Profils construits pour 631,986 utilisateurs


user_id,user_idx,total_ratings,avg_rating,std_rating,max_rating,min_rating,last_activity
AFJBKPK5W56XWSNPQU2WW66ISWYQ,88,23,4.565217391304348,0.5897678246195887,5,3,2022-11-18T18:47:03.375Z
AETOPBLLOVEAMSUYLIVTIKRWTGZA,165200,1,5.0,null,5,5,2021-08-21T23:21:39.167Z
AEOVCZC77QZJQPBIAIKCFV7AS7PA,143700,1,5.0,null,5,5,2014-08-14T11:35:14.000Z
AGZZXSMMS4WRHHJRBUJZI4FZDHKQ,127,18,5.0,0.0,5,5,2021-09-05T19:09:35.188Z
AG4BMBZUKXSF2OVXVXVYALJ7DEWQ,350067,1,2.0,null,2,2,2022-07-09T20:31:36.554Z
AGWIU2DVVALEVN4DNKJ5CIBFU4JQ,469483,1,5.0,null,5,5,2019-02-20T04:46:04.988Z
AET4ADL3QEOPDPFISNCIJKHJYWRQ,162613,1,5.0,null,5,5,2019-05-15T20:45:53.945Z
AH6ZFONXE2BNKUNRVJHNSMGXJRVA,508617,1,5.0,null,5,5,2019-09-10T13:42:02.292Z
AF2TG7OJY433QAKUCSHFFTL3RHRQ,197755,1,5.0,null,5,5,2022-01-01T12:46:05.336Z
AFWRWOLHIYLL3TSLN6YRRB3JZIJQ,325127,1,5.0,null,5,5,2019-12-08T06:19:10.507Z


Construire les profils produits

In [0]:
# Profil de chaque produit
product_profiles = ratings_indexed.groupBy("parent_asin", "item_idx").agg(
    count("*").alias("total_ratings"),          # popularité
    avg("rating").alias("avg_rating"),           # qualité perçue
    stddev("rating").alias("std_rating")         # controversialité
)

print(f" Profils construits pour {product_profiles.count():,} produits")
display(product_profiles.limit(10))

 Profils construits pour 112,565 produits


parent_asin,item_idx,total_ratings,avg_rating,std_rating
B01AKTGHFW,4943,23,3.8260869565217392,1.6418441381303657
B00QZEBH5W,19910,6,3.1666666666666665,1.834847859269718
B00J4YYA86,28159,4,4.75,0.4999999999999999
B07MZT83KK,14137,9,4.333333333333333,1.118033988749895
B08GJJ5RV9,5078,23,4.434782608695652,0.9920633665850978
B086SSHLWT,6331,19,3.6315789473684212,1.4609938138050442
B08NTD1NM1,5088,23,4.391304347826087,0.98807114368361
B08DD6BFFM,2587,40,4.125,1.4533339214301326
B08B83GH77,59473,2,4.0,0.0
B07D487TV7,10360,12,3.8333333333333335,1.7494587907710375


Sauvegarder tout dans le Volume

In [0]:
VOLUME = "/Volumes/workspace/default/ecommerce_data"

# Sauvegarder ratings indexés (utilisé par ALS)
ratings_indexed.write \
    .format("parquet") \
    .mode("overwrite") \
    .save(f"{VOLUME}/ratings_indexed")

# Sauvegarder profils utilisateurs
user_profiles.write \
    .format("parquet") \
    .mode("overwrite") \
    .save(f"{VOLUME}/user_profiles")

# Sauvegarder profils produits
product_profiles.write \
    .format("parquet") \
    .mode("overwrite") \
    .save(f"{VOLUME}/product_profiles")

print(" Tout est sauvegardé !")
print(f"\n Contenu du  Volume :")
print(f"  → ratings_indexed  : données prêtes pour ALS")
print(f"  → user_profiles    : profils utilisateurs")
print(f"  → product_profiles : profils produits")
print(f"  → user_mapping     : index → user_id")
print(f"  → item_mapping     : index → product_id")

 Tout est sauvegardé !

 Contenu du  Volume :
  → ratings_indexed  : données prêtes pour ALS
  → user_profiles    : profils utilisateurs
  → product_profiles : profils produits
  → user_mapping     : index → user_id
  → item_mapping     : index → product_id


Vérification 

In [0]:
# Recharger et vérifier
ratings_idx   = spark.read.parquet(f"{VOLUME}/ratings_indexed")
user_prof     = spark.read.parquet(f"{VOLUME}/user_profiles")
product_prof  = spark.read.parquet(f"{VOLUME}/product_profiles")


print(" FEATURE ENGINEERING TERMINÉ")

print(f" Ratings indexés  : {ratings_idx.count():,}")
print(f" Profils users    : {user_prof.count():,}")
print(f" Profils produits : {product_prof.count():,}")


 FEATURE ENGINEERING TERMINÉ
 Ratings indexés  : 701,528
 Profils users    : 631,986
 Profils produits : 112,565
